# Laboratorio 14.5 · Una red que lee números escritos a mano
**Redes Neuronales en Mortales**, Capítulo II, sección 14.5

Una cooperativa quinuera del altiplano guarda años de planillas escritas a mano: cuántos quintales entregó cada familia, en qué fecha, a qué precio. Pasarlas a la computadora a mano tomaría meses. ¿Puede ayudar una red neuronal?

En este cuaderno vas a usar una red ya entrenada como se usaría en la vida real: **guardarla, volver a cargarla, pedirle respuestas sobre números que nunca vio, leer su confianza y decidir cuándo creerle y cuándo pedir ayuda a una persona.**

**Cómo usarlo:** ejecuta cada celda en orden con ▶ (o Mayúsculas + Enter). Si Colab avisa que el cuaderno no fue creado por Google, elige «Ejecutar de todos modos».

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

digitos = load_digits()   # 1 797 dígitos reales escritos a mano (Alpaydin y Kaynak, 1998)
X, y = digitos.images, digitos.target
print("Imágenes:", X.shape[0], "| tamaño de cada una:", X.shape[1], "×", X.shape[2], "píxeles")

## 1 · Así se ven los datos
Cada dígito es una cuadrícula de 8 × 8 = 64 números entre 0 (blanco) y 16 (negro). Para la red, una imagen es solo una lista de 64 números.

In [ ]:
fig, ejes = plt.subplots(1, 10, figsize=(12, 1.6))
for k, eje in enumerate(ejes):
    eje.imshow(X[y == k][0], cmap="gray_r"); eje.set_title(str(k)); eje.axis("off")
plt.show()

## 2 · Entrenar (rápido) y apartar lo que la red nunca verá
El entrenamiento se explica en la sección 13. Aquí lo hacemos en una línea, porque lo que nos interesa es **usar** la red. Lo importante: el 25 % de los dígitos queda guardado bajo llave. La red nunca los verá mientras aprende.

In [ ]:
X_entr, X_nuevos, y_entr, y_nuevos = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)

red = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=1)
red.fit(X_entr.reshape(len(X_entr), -1) / 16, y_entr)
print("Entrenada con", len(X_entr), "dígitos. Guardamos", len(X_nuevos), "que nunca vio.")

## 3 · Guardar y volver a cargar (sección 14.2)
Lo aprendido vive en los pesos. Los guardamos en un archivo, lo volvemos a abrir y hacemos una **prueba de humo**: la red cargada debe responder exactamente lo mismo que la original. Los formatos profesionales se explican en la sección 15.1.

In [ ]:
joblib.dump(red, "red_digitos.joblib")
red_cargada = joblib.load("red_digitos.joblib")

muestra = X_nuevos[:20].reshape(20, -1) / 16
iguales = np.allclose(red.predict_proba(muestra), red_cargada.predict_proba(muestra))
print("¿La red cargada responde igual que la original?", "Sí ✅" if iguales else "No ❌")

## 4 · Usar la red con dígitos que nunca vio (secciones 14.1 y 14.3)
Para usarla, cada imagen se prepara **igual** que en el entrenamiento (aquí, dividir entre 16) y se le pide una respuesta. La red no aprende nada nuevo mientras responde.

In [ ]:
entrada = X_nuevos.reshape(len(X_nuevos), -1) / 16
probabilidades = red_cargada.predict_proba(entrada)
prediccion = probabilidades.argmax(axis=1)
confianza = probabilidades.max(axis=1)

aciertos = (prediccion == y_nuevos).sum()
print(f"Acertó {aciertos} de {len(y_nuevos)} dígitos que nunca vio ({aciertos/len(y_nuevos)*100:.1f} %)")

fig, ejes = plt.subplots(1, 8, figsize=(12, 2))
for i, eje in enumerate(ejes):
    eje.imshow(X_nuevos[i], cmap="gray_r"); eje.axis("off")
    eje.set_title(f"dice {prediccion[i]}\n{confianza[i]*100:.0f} %", fontsize=10)
plt.show()

## 5 · ¿Cuánto vale su confianza? (sección 14.4)
Agrupamos las respuestas según la confianza que mostró la red y contamos cuántas fueron correctas en cada grupo.

In [ ]:
rangos = [(0, .6), (.6, .8), (.8, .95), (.95, .99), (.99, 1.01)]
print(f"{'confianza':>12} | {'respuestas':>10} | {'aciertos':>8}")
for a, b in rangos:
    k = (confianza >= a) & (confianza < b)
    if k.sum():
        print(f"{a*100:4.0f}–{min(b,1)*100:3.0f} % | {k.sum():10d} | {(prediccion[k] == y_nuevos[k]).mean()*100:7.1f} %")

### Los errores más seguros
Los errores con baja confianza son fáciles de atrapar. Los peligrosos son los que la red comete **muy segura**.

In [ ]:
errores = np.where(prediccion != y_nuevos)[0]
errores = errores[np.argsort(-confianza[errores])]
print("Errores:", len(errores), "| con más de 90 % de confianza:", (confianza[errores] > 0.9).sum())
fig, ejes = plt.subplots(1, min(6, len(errores)), figsize=(10, 2))
for eje, i in zip(np.atleast_1d(ejes), errores[:6]):
    eje.imshow(X_nuevos[i], cmap="gray_r"); eje.axis("off")
    eje.set_title(f"era {y_nuevos[i]}, dijo {prediccion[i]}\n{confianza[i]*100:.0f} %", fontsize=9)
plt.show()

## 6 · Lo que de verdad nunca vio (sección 14.1)
Hasta ahora le mostramos dígitos nuevos, pero **parecidos** a los del entrenamiento. ¿Qué pasa si le mostramos algo que no es un dígito? Ruido al azar, una hoja en blanco, un cuadro todo negro…

In [ ]:
rng = np.random.default_rng(0)
ruido = rng.uniform(0, 16, (500, 64))
p_ruido = red_cargada.predict_proba(ruido / 16)
print(f"Ruido al azar: confianza promedio {p_ruido.max(axis=1).mean()*100:.0f} %")
print(f"Ruido al azar: el {(p_ruido.max(axis=1) > 0.9).mean()*100:.0f} % de las veces dice un número con más de 90 % de confianza")

for nombre, imagen in [("hoja en blanco", np.zeros(64)), ("cuadro negro", np.full(64, 16.0))]:
    p = red_cargada.predict_proba([imagen / 16])[0]
    print(f"{nombre}: dice {p.argmax()} con {p.max()*100:.0f} % de confianza")

La red **no tiene una respuesta «esto no es un número»**. Siempre elige uno de los diez dígitos, y a veces con mucha seguridad. Por eso la confianza alta no garantiza que la entrada tenga sentido.

## 7 · Una regla para decidir: ¿cuándo creerle?
En la cooperativa no hace falta que la red lea todo. Basta con que lea bien lo que lee, y que lo dudoso pase a una persona. Probemos varios umbrales de confianza.

In [ ]:
print(f"{'umbral':>7} | {'lee sola':>9} | {'aciertos en lo que lee':>23} | {'pasa a una persona':>18}")
for u in [0.5, 0.8, 0.9, 0.99]:
    lee = confianza >= u
    print(f"{u*100:6.0f} % | {lee.mean()*100:8.1f} % | {(prediccion[lee] == y_nuevos[lee]).mean()*100:22.1f} % | {(~lee).sum():18d}")

Subir el umbral hace que la red lea menos planillas sola, pero se equivoque menos en las que lee. **La decisión de dónde poner el umbral no es técnica: depende de cuánto cuesta un error y cuánto tiempo tiene la persona que revisa.**

## Retos para ti
1. Cambia `hidden_layer_sizes=(64,)` por `(16,)`. ¿Baja mucho el acierto?
2. En el paso 7, ¿qué umbral elegirías si cada error significa pagarle mal a una familia productora?
3. Dibuja tu propio «dígito» como una lista de 64 números (por ejemplo, una letra A) y pregúntale a la red. ¿Qué responde?
4. **Para pensar:** si la cooperativa escribe los sietes con una rayita en el medio y la red aprendió con sietes sin rayita, ¿qué pasará? ¿Cómo lo detectarías?

---
*Datos: Alpaydin, E. y Kaynak, C. (1998). Optical Recognition of Handwritten Digits. UCI Machine Learning Repository. Incluidos en Scikit-learn.*